# U-Net Architecture Comparison

Trains five configurations on the same data split, in a cumulative progression:

| Config | Loss | Residual | Dropout |
|---|---|---|---|
| `BCEDice` | BCE + Dice | ✗ | ✗ |
| `FocalDice` | Focal + Dice | ✗ | ✗ |
| `+ residual` | Focal + Dice | ✓ | ✗ |
| `+ dropout` | Focal + Dice | ✗ | ✓ |
| `+ both` | Focal + Dice | ✓ | ✓ |

**Quick smoke-test**: `MAX_TILES = 500`, `MAX_EPOCHS = 3`  
**Real comparison**: `MAX_TILES = None`, `MAX_EPOCHS = 20`

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
import time
from collections import OrderedDict
from pathlib import Path
import sys, os

PROJECT_ROOT = Path().resolve().parents[0]
sys.path.insert(0, str(PROJECT_ROOT))

from torch import nn
from torch.utils.data import Dataset, DataLoader

from lunar_segmentation.lunar_segmentation.models.unet import SmallUNet
from lunar_segmentation.lunar_segmentation.training.trainer import (
    Trainer, FocalDiceLoss, BCEDiceLoss, multilabel_metrics
)
from lunar_segmentation.lunar_segmentation.data.preprocessing import CLASS_NAMES

print("Imports OK")

## 1. Experiment Controls

Change only this cell to adjust the experiment scope.

In [ ]:
# ─── HOW MUCH DATA / TIME ─────────────────────────────────────────────────────
MAX_TILES  = 1000   # int → first N tiles; None → full 15 931-tile dataset
MAX_EPOCHS = 5      # 3-5 for a quick smoke-test; 20 for a real comparison
BATCH_SIZE = 16
VAL_SPLIT  = 0.2    # fraction held out for validation (same split for all models)
# ──────────────────────────────────────────────────────────────────────────────

# ─── LOSS ─────────────────────────────────────────────────────────────────────
LR            = 1e-3
CLASS_WEIGHTS = torch.tensor([1.0, 4.0, 5.0, 5.0, 5.0, 5.0, 5.0])
# ──────────────────────────────────────────────────────────────────────────────

BASE_DIR   = Path().resolve().parents[1] / "data" / "MR"
INDEX_PATH = BASE_DIR / "tiles" / "index.csv"

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Device     : {DEVICE}")
print(f"MAX_TILES  : {MAX_TILES}")
print(f"MAX_EPOCHS : {MAX_EPOCHS}")
print(f"BATCH_SIZE : {BATCH_SIZE}")

## 2. Data

In [ ]:
df = pd.read_csv(INDEX_PATH)
# Resolve relative paths to absolute
df["tile_path"] = df["tile_path"].apply(lambda p: str(BASE_DIR / p))

if MAX_TILES is not None:
    df = df.iloc[:MAX_TILES].reset_index(drop=True)

print(f"Using {len(df)} tiles from {df.aoi.nunique()} AOI(s)")

# Load all tiles into RAM once — shared across all model runs
print(f"Loading {len(df)} tiles into RAM…", flush=True)
_tiles = []
for _, row in df.iterrows():
    data = np.load(row["tile_path"])
    _tiles.append((data["image"].astype(np.float32),
                   data["mask"].astype(np.float32)))
print("Done.")

# Train / val split (fixed seed → same for every model)
rng     = np.random.default_rng(42)
idx     = rng.permutation(len(_tiles))
n_val   = max(1, int(len(idx) * VAL_SPLIT))
val_idx, train_idx = idx[:n_val], idx[n_val:]
print(f"Train: {len(train_idx)}  Val: {len(val_idx)}")

In [ ]:
class _TileDataset(Dataset):
    """Thin wrapper around a pre-loaded list of (image, mask) arrays."""
    def __init__(self, tiles, indices, augment=False):
        self.tiles   = tiles
        self.indices = indices
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, mask = self.tiles[self.indices[i]]
        img, mask = img.copy(), mask.copy()
        if self.augment and np.random.random() < 0.5:
            img  = img[:,  :, ::-1].copy()
            mask = mask[:, :, ::-1].copy()
        return torch.from_numpy(img), torch.from_numpy(mask)

train_ds = _TileDataset(_tiles, train_idx, augment=True)
val_ds   = _TileDataset(_tiles, val_idx,   augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Loaders ready  ({len(train_loader)} train batches, {len(val_loader)} val batches)")

## 3. Model Configurations

In [ ]:
# Each entry: (display_name, arch_kwargs, loss_tag)
# loss_tag "bce_dice"   → BCEDiceLoss  (no class weights)
# loss_tag "focal_dice" → FocalDiceLoss(gamma=2, alpha=0.25, class_weights)
CONFIGS = OrderedDict([
    ("BCEDice",      dict(use_residual=False, dropout=0.0, loss="bce_dice")),
    ("FocalDice",    dict(use_residual=False, dropout=0.0, loss="focal_dice")),
    ("+ residual",   dict(use_residual=True,  dropout=0.0, loss="focal_dice")),
    ("+ dropout",    dict(use_residual=False, dropout=0.3, loss="focal_dice")),
    ("+ both",       dict(use_residual=True,  dropout=0.3, loss="focal_dice")),
])

def make_criterion(loss_tag):
    if loss_tag == "bce_dice":
        return BCEDiceLoss()
    return FocalDiceLoss(gamma=2.0, alpha=0.25, class_weights=CLASS_WEIGHTS)

# Architecture + loss summary
print(f"{'Config':<14} {'Loss':<12} {'Params':>10}  Notes")
print("-" * 55)
for name, cfg in CONFIGS.items():
    arch = {k: v for k, v in cfg.items() if k != "loss"}
    m    = SmallUNet(in_channels=3, num_classes=len(CLASS_NAMES), **arch)
    notes = ("residual " if arch["use_residual"] else "") +             (f"dropout={arch['dropout']}" if arch["dropout"] > 0 else "")
    print(f"{name:<14} {cfg['loss']:<12} {m.count_parameters():>10,}  {notes or '—'}")

## 4. Train All Configs

In [ ]:
results = {}

for name, cfg in CONFIGS.items():
    print(f"\n{'='*50}\n  Training: {name}\n{'='*50}")

    arch_kwargs = {k: v for k, v in cfg.items() if k != "loss"}
    model       = SmallUNet(in_channels=3, num_classes=len(CLASS_NAMES), **arch_kwargs)
    criterion   = make_criterion(cfg["loss"])
    optimizer   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
                      optimizer, T_max=MAX_EPOCHS, eta_min=1e-5)
    trainer     = Trainer(model, optimizer, criterion, device=DEVICE, scheduler=scheduler)

    train_losses, epoch_times = [], []
    for epoch in range(1, MAX_EPOCHS + 1):
        t0   = time.time()
        loss = trainer.train_one_epoch(train_loader)
        dt   = time.time() - t0
        train_losses.append(loss)
        epoch_times.append(dt)
        print(f"  epoch {epoch:>2}/{MAX_EPOCHS}  loss={loss:.4f}  "
              f"lr={trainer.current_lr():.2e}  {dt:.1f}s")

    results[name] = dict(
        train_losses = train_losses,
        epoch_times  = epoch_times,
        val_metrics  = trainer.evaluate(val_loader),
        n_params     = model.count_parameters(),
        loss_tag     = cfg["loss"],
    )

print("\nAll configs trained.")

## 5. Results
### 5.1 Training Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for name, r in results.items():
    ax.plot(range(1, MAX_EPOCHS + 1), r["train_losses"], marker="o", label=name, linewidth=2)

ax.set_xlabel("Epoch")
ax.set_ylabel("Train loss (FocalDice)")
ax.set_title("Training loss — all configs")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / "results/comparison_train_loss.png", dpi=130)
plt.show()

### 5.2 Validation Metrics per Class

In [ ]:
metric_rows = []
for name, r in results.items():
    if r["val_metrics"] is not None:
        df_m = r["val_metrics"].copy()
        df_m["config"] = name
        metric_rows.append(df_m.reset_index())

if not metric_rows:
    print("No validation metrics to plot.")
else:
    all_metrics = pd.concat(metric_rows, ignore_index=True)
    n_configs   = len(CONFIGS)

    for metric in ("f1", "iou"):
        fig, ax = plt.subplots(figsize=(14, 4))
        width   = 0.14
        x       = np.arange(len(CLASS_NAMES))
        offsets = np.linspace(-(n_configs - 1) / 2,
                               (n_configs - 1) / 2, n_configs) * width

        # Separate loss configs from arch configs visually
        colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
        for offset, color, (name, _) in zip(offsets, colors, CONFIGS.items()):
            vals = all_metrics[all_metrics["config"] == name].set_index("class")[metric]
            vals = [float(vals.get(c, 0)) for c in CLASS_NAMES]
            ax.bar(x + offset, vals, width, label=name, color=color, alpha=0.85)

        # Vertical divider between loss comparison and arch comparison
        ax.axvline(x=0 - width * 3, color="grey", linestyle=":", linewidth=1, alpha=0.5)

        ax.set_xticks(x)
        ax.set_xticklabels(CLASS_NAMES, rotation=25, ha="right", fontsize=9)
        ax.set_ylabel(metric.upper())
        ax.set_title(f"Validation {metric.upper()} per class")
        ax.legend(fontsize=9, ncol=n_configs)
        ax.set_ylim(0, 1)
        ax.grid(True, axis="y", alpha=0.3)
        plt.tight_layout()
        plt.savefig(BASE_DIR / f"results/comparison_val_{metric}.png", dpi=130)
        plt.show()

### 5.3 Summary Table

In [ ]:
rows = []
for name, r in results.items():
    row = {
        "config":         name,
        "params":         f"{r['n_params']:,}",
        "final_loss":     f"{r['train_losses'][-1]:.4f}",
        "avg_epoch_s":    f"{np.mean(r['epoch_times']):.1f}s",
    }
    if r["val_metrics"] is not None:
        row["mean_F1"]  = f"{r['val_metrics']['f1'].mean():.4f}"
        row["mean_IoU"] = f"{r['val_metrics']['iou'].mean():.4f}"
    else:
        row["mean_F1"] = row["mean_IoU"] = "—"
    rows.append(row)

summary = pd.DataFrame(rows).set_index("config")
print(summary.to_string())
print()
print("Best mean F1 :", summary["mean_F1"].idxmax()  if "mean_F1" in summary else "n/a")
print("Best mean IoU:", summary["mean_IoU"].idxmax() if "mean_IoU" in summary else "n/a")